In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path
import pickle

from config import (
    TAGS_EXCEL_PATH, DATA_CSV_PART1, DATA_CSV_PART2, TARGET_COL,
    load_tag_lists,
)

# Квантильная агрегация: alcohol

AggregatedHashtagsProcessor (MinMax + quantile binning).

In [ ]:
tags_descriptions = pd.read_excel(TAGS_EXCEL_PATH, sheet_name='HT_list')
tag_lists = load_tag_lists(tags_descriptions)

alcohol_features_list = tag_lists['alcohol_features_list']
youth_features_list = tag_lists['youth_features_list']
middle_features_list = tag_lists['middle_features_list']
mature_features_list = tag_lists['mature_features_list']
elderly_features_list = tag_lists['elderly_features_list']

part1 = pd.read_csv(DATA_CSV_PART1, encoding='cp1251', delimiter=',')
part2 = pd.read_csv(DATA_CSV_PART2, encoding='cp1251', delimiter=',')
casco_hashtags_full = pd.merge(part1, part2, on='POLICY_ZV', how='inner')
casco_hashtags_full[TARGET_COL] = casco_hashtags_full['CLAIMS_PART_DAM_COUNT'].astype(bool).astype(int)
casco_hashtags_full.set_index('POLICY_ZV', inplace=True)

casco_hashtags_full.drop(columns=['TAG_JOIN_IND'], inplace=True)
casco_hashtags_full['SUM'] = casco_hashtags_full.filter(like='TAG_').fillna(0).sum(axis=1)
casco_hashtags_full = casco_hashtags_full[casco_hashtags_full['SUM'] > 0]
print('shape:', casco_hashtags_full.shape)

## AggregatedHashtagsProcessor

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import pickle

class AggregatedHashtagsProcessor:
    """
    Выполняет агрегацию признаков по тематическим группам и формирует единую шкалу возраста.
    
    Логика работы:
    1. Тематическая агрегация: Суммирует признаки из групп (напр., 'alcohol') и нормирует 
       их в диапазон [0, 1] через MinMaxScaler. Это позволяет сравнивать интенсивность 
       интересов разного объема.
    2. Возрастной индекс (Ordinal Encoding): Объединяет разрозненные признаки возрастных 
       групп в единую непрерывную шкалу от 0.0 до 1.0. 
       - 0.00: Youth (Молодежь)
       - 0.33: Middle (Средний возраст)
       - 0.66: Mature (Зрелость)
       - 1.00: Elderly (Пожилые)
    
    Интерпретация:
    - Значения тематических групп: степень выраженности интереса относительно всей выборки.
    - Значение agg_age_index: поведенческий возраст пользователя (чем ближе к 1, тем "старше" 
      набор хэштегов).
    
    Args:
        df (pd.DataFrame): Исходный датафрейм с признаками-хэштегами.
        feature_dict (dict): Словарь {'Имя_группы': [список_колонок]} для обычных интересов.
        age_groups (list of lists): Список из 4-х списков колонок строго в порядке: 
                                    [youth, middle, mature, elderly].
         n_quantiles (int): Количество квантилей для бининга (по умолчанию 20).
    
    Returns:
        pd.DataFrame: Таблица с новыми агрегированными признаками.
    """
    def __init__(self, n_quantiles=20):
        self.n_quantiles = n_quantiles
        self.scalers_ = {}
        self.bins_ = {}
        self.is_fitted = False
        self.feature_dict_ = None
        self.age_groups_ = None

    def _safe_column_selection(self, df, cols):
        """Безопасно выбирает колонки, игнорируя отсутствующие."""
        existing_cols = [col for col in cols if col in df.columns]
        missing_cols = set(cols) - set(existing_cols)
        if missing_cols:
            print(f"Внимание: следующие колонки не найдены: {missing_cols}")
        return existing_cols

    def fit(self, df, feature_dict, age_groups=None):
        """
        Обучает процессор с проверкой существования колонок.
        """
        self.feature_dict_ = {k: list(v) for k, v in feature_dict.items()}
        self.age_groups_ = [list(g) for g in age_groups] if age_groups else None

        for agg_name, cols in self.feature_dict_.items():
            selected_cols = self._safe_column_selection(df, cols)
            if not selected_cols:
                raise ValueError(f"Ни одна из колонок для '{agg_name}' не найдена в данных.")

            sum_series = df[selected_cols].sum(axis=1, min_count=1).fillna(0)
            
            scaler = MinMaxScaler()
            normalized = scaler.fit_transform(sum_series.values.reshape(-1, 1)).flatten()
            self.scalers_[agg_name] = scaler

            quantiles = np.linspace(0, 1, self.n_quantiles + 1)
            bins = np.quantile(normalized, quantiles)
            bins = np.unique(bins)
            self.bins_[agg_name] = bins

        # Обработка возрастных групп
        if self.age_groups_ and len(self.age_groups_) == 4:
            group_names = ['youth', 'middle', 'mature', 'elderly']
            selected_age_groups = []
            for i, (group_cols, name) in enumerate(zip(self.age_groups_, group_names)):
                selected = self._safe_column_selection(df, group_cols)
                if not selected:
                    print(f"Внимание: нет колонок для возрастной группы '{name}'. Будет проигнорирована.")
                selected_age_groups.append(selected)

            if not any(selected_age_groups):
                raise ValueError("Не найдено ни одной возрастной колонки.")

            y_cols, mid_cols, mat_cols, eld_cols = selected_age_groups
            s_y = df[y_cols].sum(axis=1) if y_cols else 0
            s_mid = df[mid_cols].sum(axis=1) if mid_cols else 0
            s_mat = df[mat_cols].sum(axis=1) if mat_cols else 0
            s_eld = df[eld_cols].sum(axis=1) if eld_cols else 0

            total = s_y + s_mid + s_mat + s_eld
            total = total.replace(0, 1)  # избегаем деления на 0

            age_index = (s_y * 0.0 + s_mid * 0.33 + s_mat * 0.66 + s_eld * 1.0) / total
            age_index = age_index.fillna(0).values

            scaler_age = MinMaxScaler()
            age_norm = scaler_age.fit_transform(age_index.reshape(-1, 1)).flatten()
            self.scalers_['agg_age_index'] = scaler_age

            quantiles = np.linspace(0, 1, self.n_quantiles + 1)
            bins = np.quantile(age_norm, quantiles)
            bins = np.unique(bins)
            self.bins_['agg_age_index'] = bins

        self.is_fitted = True
        return self

    def transform(self, df):
        if not self.is_fitted:
            raise ValueError("Processor должен быть обучен сначала с помощью .fit()")

        res_df = pd.DataFrame(index=df.index)

        for agg_name, cols in self.feature_dict_.items():
            selected_cols = self._safe_column_selection(df, cols)
            if not selected_cols:
                # Если нет колонок — заполняем нулями (метка 0 — самый низкий уровень)
                res_df[agg_name] = 0
                continue

            sum_series = df[selected_cols].sum(axis=1, min_count=1).fillna(0)
            normalized = self.scalers_[agg_name].transform(sum_series.values.reshape(-1, 1)).flatten()

            # Применяем сохранённые бины
            res_df[agg_name] = np.digitize(normalized, self.bins_[agg_name]) - 1
            res_df[agg_name] = np.clip(res_df[agg_name], 0, self.n_quantiles - 1)

        # Возрастной индекс
        if self.age_groups_ and 'agg_age_index' in self.scalers_:
            group_names = ['youth', 'middle', 'mature', 'elderly']
            selected_age_groups = []
            for group_cols, name in zip(self.age_groups_, group_names):
                selected = self._safe_column_selection(df, group_cols)
                selected_age_groups.append(selected)

            y_cols, mid_cols, mat_cols, eld_cols = selected_age_groups
            s_y = df[y_cols].sum(axis=1) if y_cols else 0
            s_mid = df[mid_cols].sum(axis=1) if mid_cols else 0
            s_mat = df[mat_cols].sum(axis=1) if mat_cols else 0
            s_eld = df[eld_cols].sum(axis=1) if eld_cols else 0

            total = s_y + s_mid + s_mat + s_eld
            total = total.replace(0, 1)

            age_index = (s_y * 0.0 + s_mid * 0.33 + s_mat * 0.66 + s_eld * 1.0) / total
            age_index = age_index.fillna(0).values

            age_norm = self.scalers_['agg_age_index'].transform(age_index.reshape(-1, 1)).flatten()
            res_df['agg_age_index'] = np.digitize(age_norm, self.bins_['agg_age_index']) - 1
            res_df['agg_age_index'] = np.clip(res_df['agg_age_index'], 0, self.n_quantiles - 1)
        else:
            res_df['agg_age_index'] = 0  # дефолт, если не обучался

        return res_df

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump(self, f)

    @staticmethod
    def load(path):
        with open(path, 'rb') as f:
            return pickle.load(f)

In [ ]:
ARTIFACTS_DIR = Path('artifacts')
CSV_DIR = ARTIFACTS_DIR / 'csv'
MODELS_DIR = ARTIFACTS_DIR / 'models'
for p in (CSV_DIR, MODELS_DIR):
    p.mkdir(parents=True, exist_ok=True)

other_groups = {
    'agg_alcohol': alcohol_features_list,
}

age_lists = [
    youth_features_list,
    middle_features_list,
    mature_features_list,
    elderly_features_list,
]

processor = AggregatedHashtagsProcessor(n_quantiles=30)
processor.fit(casco_hashtags_full, other_groups, age_groups=age_lists)
processor.save(MODELS_DIR / 'aggregated_hashtags_processor_alcohol.pkl')

new_features = processor.transform(casco_hashtags_full)
new_features.to_csv(CSV_DIR / 'aggregated_hashtags_alcohol.csv')
print(new_features.head())

## Анализ связи alcohol с claim rate

In [ ]:
check_df = pd.concat([new_features, casco_hashtags_full[TARGET_COL]], axis=1)

analysis = check_df.groupby('agg_alcohol')[TARGET_COL].agg(['mean', 'count'])
analysis.columns = ['claim_rate', 'objects']
print(analysis)

analysis['claim_rate'].plot(kind='bar', figsize=(10, 5), color='steelblue')
plt.title('Связь agg_alcohol с частотой аварий')
plt.ylabel('Средняя частота аварий')
plt.xlabel('Группа интенсивности (agg_alcohol)')
plt.tight_layout()
plt.show()